# Zrozumienie mechanizmu działania Transformerów w LLM

**Instrukcja wykonania krok po kroku**

Ćwiczenie realizujecie w przygotowanym notebooku z załączenia do postu.

## Krok 1: Tokenizacja

- **Zadanie**: Uruchomcie komórki w sekcji 1. Zmieńcie zdanie wejściowe na własny przykład (np. zdanie z polskimi znakami lub specjalistyczną terminologią).
- **Cel**: Zrozumienie, jak tekst jest konwertowany na ID tokenów i dlaczego niektóre słowa są dzielone na części.

## Krok 2: Mechanizm Self-Attention

- **Zadanie**: Wykonajcie sekcje 2 i 3. Wypełnijcie macierz wag w zadaniu 2C i wykonajcie ręczne obliczenia w sekcji 3.
- **Cel**: Zrozumienie roli macierzy Q, K i V oraz funkcji Softmax w obliczaniu ważności kontekstu dla każdego tokena.

## Krok 3: Generowanie i Causal Mask

- **Zadanie**: Przeanalizujcie sekcję 4 dotyczącą maskowania (Causal Mask). Uruchomcie generowanie tekstu przy użyciu modelu `distilgpt2` i eksperymentujcie z parametrem `temperature`.
- **Cel**: Zrozumienie, jak model przewiduje kolejny token i jak "temperatura" wpływa na rozkład prawdopodobieństwa (kreatywność vs przewidywalność).

# Ćwiczenie: Jak działają transformery w modelach LLM
 
**Cel:** zrozumienie tokenizacji, self-attention, maskowania oraz generowania token po tokenie w modelach transformerowych.

## Efekty uczenia
Po wykonaniu ćwiczenia student potrafi:
- wyjaśnić rolę tokenizacji i embeddingów,
- opisać intuicję działania self-attention,
- odróżnić attention pełny od attention maskowanego (causal),
- uruchomić prosty eksperyment z modelem transformers,
- zinterpretować wpływ parametrów generowania na wynik modelu.


## Instrukcja pracy
1. Wykonuj komórki **po kolei**.
2. Nie przechodź dalej, dopóki nie uzupełnisz odpowiedzi tekstowych.
3. Tam, gdzie widzisz `TODO`, wpisz własną odpowiedź.
4. Po każdej części zapisz krótki wniosek.
5. Na końcu zapisz notebook z uzupełnionymi odpowiedziami i oddaj go prowadzącemu.


In [ ]:
# Jeśli pracujesz w Google Colab, odkomentuj poniższą linię przy pierwszym uruchomieniu:
# !pip -q install transformers torch pandas matplotlib seaborn

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch

sns.set_theme(style='whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## Część 1. Tokenizacja i wejście do modelu (20 min)
Transformer nie operuje bezpośrednio na zdaniach, lecz na **tokenach**. W tej części sprawdzisz, jak tokenizer dzieli tekst na jednostki wejściowe.

### Zadanie 1A
Przed uruchomieniem kodu odpowiedz własnymi słowami:
- Czym różni się znak od słowa i od tokena?
- Dlaczego model nie pracuje na pełnych zdaniach jako pojedynczych obiektach?

**Twoja odpowiedź:** `TODO`


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
text = 'Transformers are changing natural language processing.'
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
pd.DataFrame({'token': tokens, 'token_id': token_ids})


### Zadanie 1B
1. Zmień wartość zmiennej `text` na własne zdanie techniczne lub potoczne.
2. Uruchom komórkę ponownie.
3. Sprawdź, czy któreś słowo zostało rozbite na kilka tokenów.

**Odpowiedz:**
- Które fragmenty zostały rozbite?
- Co to mówi o sposobie reprezentacji języka przez model?

**Twoja odpowiedź:** `TODO`


## Część 2. Intuicja self-attention na małym przykładzie
W self-attention każdy token porównuje się z innymi tokenami w sekwencji. Dzięki temu model może ocenić, które słowa są ważne przy interpretacji danego słowa.

### Zadanie 2A
Przeczytaj zdanie: **"The animal didn't cross the street because it was too tired."**

Zanim uruchomisz kod, odpowiedz:
- Do czego odnosi się słowo `it`?
- Które wcześniejsze słowa powinny mieć wysoką wagę uwagi przy interpretacji `it`?

**Twoja odpowiedź:** `TODO`


In [ ]:
sentence = "The animal didn't cross the street because it was too tired."
words = sentence.replace('.', '').split()

attention_scores = np.array([
    [1.0, 0.2, 0.1, 0.1, 0.1, 0.05, 0.05, 0.02, 0.02, 0.02],
    [0.2, 1.0, 0.1, 0.1, 0.1, 0.08, 0.08, 0.05, 0.04, 0.03],
    [0.1, 0.1, 1.0, 0.2, 0.1, 0.05, 0.05, 0.03, 0.03, 0.02],
    [0.1, 0.1, 0.2, 1.0, 0.2, 0.05, 0.05, 0.03, 0.03, 0.02],
    [0.1, 0.1, 0.1, 0.2, 1.0, 0.15, 0.1, 0.05, 0.04, 0.03],
    [0.05, 0.08, 0.05, 0.05, 0.15, 1.0, 0.3, 0.08, 0.07, 0.05],
    [0.05, 0.08, 0.05, 0.05, 0.1, 0.3, 1.0, 0.5, 0.2, 0.1],
    [0.02, 0.05, 0.03, 0.03, 0.05, 0.08, 0.5, 1.0, 0.4, 0.2],
    [0.02, 0.04, 0.03, 0.03, 0.04, 0.07, 0.2, 0.4, 1.0, 0.3],
    [0.02, 0.03, 0.02, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 1.0],
])

plt.figure(figsize=(10, 7))
sns.heatmap(attention_scores, xticklabels=words, yticklabels=words, cmap='viridis')
plt.title('Uproszczona mapa self-attention')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()


### Zadanie 2B
1. Odszukaj wiersz i kolumnę odpowiadającą tokenowi `it`.
2. Wskaż 2–3 tokeny, na które `it` zwraca największą uwagę.
3. Napisz, czy wynik zgadza się z Twoją intuicją.

**Twoja odpowiedź:** `TODO`


### Zadanie 2C
Zmodyfikuj zdanie na własny przykład z niejednoznacznym zaimkiem, np.:
- "The trophy didn't fit in the suitcase because it was too big."
- "The server restarted after it exceeded memory limits."

Następnie opisz, które tokeny **powinny** mieć wysoką wagę attention.

**Twoja odpowiedź:** `TODO`


## Część 3. Uproszczone attention krok po kroku
Teraz policzysz bardzo uproszczoną wersję attention. Nie używamy pełnego modelu, tylko mały przykład numeryczny, aby zobaczyć mechanikę obliczeń.


In [ ]:
tokens = ['Ala', 'lubi', 'kawę']
X = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0]
])

Wq = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])
Wk = np.array([
    [1.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0]
])
Wv = np.array([
    [1.0, 0.0],
    [0.5, 1.0],
    [0.0, 1.0]
])

Q = X @ Wq
K = X @ Wk
V = X @ Wv
scores = Q @ K.T / math.sqrt(K.shape[1])

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

weights = softmax(scores)
output = weights @ V

print('Q =')
print(Q)
print('\nK =')
print(K)
print('\nV =')
print(V)
print('\nAttention weights =')
print(np.round(weights, 3))
print('\nOutput =')
print(np.round(output, 3))


### Zadanie 3A
Na podstawie wyników odpowiedz:
- Co reprezentują macierze `Q`, `K` i `V`?
- Co oznaczają wartości w `Attention weights`?
- Dlaczego stosujemy funkcję `softmax`?

**Twoja odpowiedź:** `TODO`


### Zadanie 3B
Zmień jedną wartość w macierzy `X` lub jednej z macierzy wag `Wq`, `Wk`, `Wv`, a następnie uruchom obliczenia ponownie.

Opisz:
- która zmiana została wykonana,
- jak zmieniły się wagi attention,
- jak wpłynęło to na końcowy output.

**Twoja odpowiedź:** `TODO`


## Część 4. Attention maskowany i generowanie token po tokenie
Modele generatywne, takie jak GPT, nie mogą patrzeć w przyszłość. Dlatego używają **causal mask**, która blokuje dostęp do kolejnych tokenów podczas przewidywania następnego słowa.


In [ ]:
tokens = ['I', 'like', 'deep', 'learning']
n = len(tokens)
mask = np.triu(np.ones((n, n)), k=1)

plt.figure(figsize=(6, 4))
sns.heatmap(mask, annot=True, cbar=False, xticklabels=tokens, yticklabels=tokens, cmap='Reds')
plt.title('Causal mask: 1 = pozycja zablokowana')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()


### Zadanie 4A
Odpowiedz:
- Dlaczego token na pozycji 2 nie powinien widzieć tokenów z pozycji 3 i 4?
- Co by się stało, gdyby model podczas treningu mógł patrzeć na przyszłe tokeny?

**Twoja odpowiedź:** `TODO`


In [ ]:
model_name = 'distilgpt2'
tokenizer_gpt = AutoTokenizer.from_pretrained(model_name)
model_gpt = AutoModelForCausalLM.from_pretrained(model_name).to(device)

prompt = 'Artificial intelligence can'
inputs = tokenizer_gpt(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model_gpt(**inputs)

logits = outputs.logits[0, -1, :]
top_k = 10
values, indices = torch.topk(logits, top_k)
top_tokens = [tokenizer_gpt.decode([idx]) for idx in indices.tolist()]
top_scores = values.cpu().numpy()

df = pd.DataFrame({'token': top_tokens, 'logit': top_scores})
df


### Zadanie 4B
1. Sprawdź 10 najbardziej prawdopodobnych kolejnych tokenów.
2. Zmień `prompt` na własny początek zdania.
3. Porównaj, jak zmienia się lista tokenów.

**Twoja odpowiedź:** `TODO`


In [ ]:
def generate_text(prompt, temperature=1.0, max_new_tokens=30):
    input_ids = tokenizer_gpt(prompt, return_tensors='pt').input_ids.to(device)
    output_ids = model_gpt.generate(
        input_ids,
        do_sample=True,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer_gpt.eos_token_id
    )
    return tokenizer_gpt.decode(output_ids[0], skip_special_tokens=True)

prompt = 'Large language models are useful because'
for temp in [0.2, 0.7, 1.2]:
    print(f'\nTEMPERATURE = {temp}')
    print(generate_text(prompt, temperature=temp))


### Zadanie 4C
Na podstawie wyników opisz wpływ parametru `temperature`:
- Co dzieje się przy niskiej temperaturze?
- Co dzieje się przy wysokiej temperaturze?
- Kiedy w praktyce warto użyć niższej, a kiedy wyższej temperatury?

**Twoja odpowiedź:** `TODO`


## Część 5. Wnioski końcowe
Uzupełnij poniższe pytania pełnymi zdaniami.

### Pytania końcowe
1. Jaką rolę pełni tokenizacja w LLM?
2. Na czym polega mechanizm self-attention?
3. Dlaczego modele generatywne stosują causal mask?
4. Jak parametr temperature wpływa na generowanie tekstu?
5. Który fragment ćwiczenia najlepiej wyjaśnił Ci działanie transformera i dlaczego?

**Twoje odpowiedzi:** `TODO`


## Dla chętnych
Jeśli skończysz wcześniej:
- porównaj `distilgpt2` z innym małym modelem,
- sprawdź różnicę między tokenizerem BERT i GPT-2,
- przygotuj własny przykład zdania, w którym attention powinno rozstrzygać wieloznaczność.
